# KrishiDisha field disease classifier - Kaggle training

Trains a larger backbone (ConvNeXt-Tiny / EfficientNetV2-S) on the unified field manifest using a Kaggle GPU (T4 x2 or P100, 30 h/week).

**Setup on Kaggle** (Settings panel on the right):
1. Accelerator: GPU T4 x2 (or P100). Internet: ON (needed for `pip`, `git clone`, pretrained weights).
2. Add data: upload the pre-resized `disease_unified` folder from your machine as a private Kaggle dataset
   (zip `C:/Shivpratap_Singh_Official_Work/datasets/disease_unified` -> *Datasets > New Dataset*), then attach it here.
   Alternatively attach the raw Kaggle sources and run the manifest builder in this notebook (cell 4).
3. Set `REPO` to your GitHub repo URL and `BRANCH` if not `main`.

Outputs (checkpoint, ONNX, reports) are written to `/kaggle/working/models` - download them from the *Output* tab and copy into the repo's `models/` folder.

In [ ]:
REPO = "https://github.com/shivpratapsinghpanwar/KrishiDisha.ai.git"
BRANCH = "main"
ARCH = "timm:convnext_tiny"      # or timm:efficientnetv2_s, timm:efficientnet_b2
IMG_SIZE = 224
EPOCHS = 12
BATCH = 64

import os, subprocess, sys
if not os.path.exists("/kaggle/working/krishidisha"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/krishidisha"], check=True)
os.chdir("/kaggle/working/krishidisha")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "onnx", "onnxruntime", "pyyaml"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# Locate the attached unified dataset (a folder containing manifest.csv + images/)
import glob
cands = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
print(cands)
MANIFEST = cands[0] if cands else None
assert MANIFEST, "attach the disease_unified dataset (or build it in the next cell)"

In [ ]:
# Kaggle input is read-only; the trainer only reads images, so we can use it in place.
# Copying to local disk is faster for many small files:
import shutil, pathlib
src = pathlib.Path(MANIFEST).parent
dst = pathlib.Path("/kaggle/working/disease_unified")
if not dst.exists():
    shutil.copytree(src, dst)
MANIFEST = str(dst / "manifest.csv")
print(MANIFEST, sum(1 for _ in open(MANIFEST)) - 1, "rows")

In [ ]:
# (Optional) build the manifest here from raw attached sources instead of uploading the unified folder.
# Set KRISHIDISHA_DATA_ROOT to a folder that contains <source_name>/ dirs matching ml/datasets/sources.yaml.
# os.environ["KRISHIDISHA_DATA_ROOT"] = "/kaggle/working/datasets"
# !python -m ml.datasets.build_disease_manifest --out /kaggle/working/disease_unified --resize 320

In [ ]:
!python -m ml.vision.train --manifest {MANIFEST} --arch {ARCH} --img-size {IMG_SIZE} --batch-size {BATCH} \
    --epochs {EPOCHS} --ema --num-workers 4 --output /kaggle/working/models --resume --calibrate

In [ ]:
# Stage B: field-weighted fine-tune (own photos x6) at a lower learning rate
!python -m ml.vision.train --manifest {MANIFEST} --arch {ARCH} --img-size {IMG_SIZE} --batch-size {BATCH} \
    --epochs 4 --lr 1e-4 --own-weight 6 --ema --num-workers 4 --output /kaggle/working/models_stageB \
    --init /kaggle/working/models/plant_disease_model.pt --calibrate

In [ ]:
!python -m ml.vision.eval --checkpoint /kaggle/working/models_stageB/plant_disease_model.pt --manifest {MANIFEST} --output /kaggle/working/models_stageB
!python -m ml.vision.export --checkpoint /kaggle/working/models_stageB/plant_disease_model.pt --output /kaggle/working/models_stageB
print(open("/kaggle/working/models_stageB/reports/disease_eval.md").read()[:3000])